> **Jumeau Rust** de [`Complexity-08e-NF-MachinesCompteARebours-Python.ipynb`](Complexity-08e-NF-MachinesCompteARebours-Python.ipynb) — kernel `rust` (evcxr), `std` seule, aucune crate. | [Index Complexity](README.md)
# Complexity-08e — Le compte à rebours : des machines finies qui attendent une contradiction (NF, C₀)

*Greffe D de [#27537](https://github.com/jsboige/CoursIA/issues/27537) (EPIC [#20607](https://github.com/jsboige/CoursIA/issues/20607)), voisine de `Complexity-08d` (Greffe A, PR #27483) et de [Complexity-07b — Le tour des grands nombres](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb), où le castor affairé est introduit.*

**Navigation** : [Index de la série](README.md) | [Python](Complexity-08e-NF-MachinesCompteARebours-Python.ipynb)
> **Jumeau Rust** — kernel `rust` (evcxr), `std` seule, aucune crate : les mêmes cellules que le jumeau Python, avec des sorties **strictement identiques** (comparaison déclarée dans `scripts/notebook_tools/rust_conformance.d/complexity-08e-nf-machines-compte-a-rebours.yaml`).
**Public : Recherche.** Le carnet ne suppose que les carnets précédents de la série : les machines de Turing pas à pas (01), la notion de réduction (02), le castor affairé et sa croissance (07b).

> **Statut épistémique.** Tout ce qui suit distingue trois natures de résultats : un **théorème Lean formalisé et audité**, un **calcul reproductible vérifié par simulateur indépendant**, et une **borne conditionnelle**. Les artefacts du dépôt `wlad-svennik/notes` sont **cités et épinglés par révision** ; aucun contenu n'est recopié dans ce dépôt (aucune licence détectée dans le dépôt source au 20/11/2026, demande de clarification ouverte : `wlad-svennik/notes#1`). La section de mécanique exécute **notre propre** machine-jouet, déclarée comme telle : elle reproduit le *câblage* du compte à rebours, jamais les machines réelles.

## 0. Trois statuts pour une même famille d'objets

Le dépôt source publie une famille de machines à nombre d'états décroissant (210, 201, 192, 185, 184) et, en parallèle, des machines calibrées sur une autre théorie (198 états pour C₀). Avant de citer un seul chiffre, la règle du carnet 08d s'applique ici mot pour mot : **classer l'objet avant de l'affirmer**.

1. **Théorème Lean formalisé et audité.** Pour `nf-bb185`, l'équivalence est un théorème non borné, `NFMachine185.halts_iff_nf_refutation`, accompagné d'un audit d'axiomes enregistré : trois axiomes standard de Lean et onze vérifications finies par évaluation native. Pour `nf-bb192`, l'équivalence a été vérifiée par Lean 4.35.0 sur la table littérale. C'est le statut le plus fort de la famille.
2. **Calcul reproductible.** Pour `nf-bb184`, le dépôt publie la table de la machine (empreinte SHA-256 annoncée `7764cc84…8798`), un simulateur indépendant (142 608 essais, 88 555 587 pas binaires), la reconstruction des 2 809 instructions émettrices d'axiomes, un point de contrôle de vérification de 1,84 Mo et un `SHA256SUMS`. Ce n'est pas un théorème *sur* NF : c'est la vérification finie d'une table.
3. **Borne conditionnelle.** Pour `c0-198`, le dépôt relie des machines de 198 états à l'incohérence de C₀ et énonce des bornes de la forme Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity). Une borne conditionnelle se cite **avec sa condition**, jamais comme un fait nu.

Le reste du carnet suit cet ordre : le mécanisme d'abord (§2), les artefacts ensuite (§3), la chaîne de compression comme méthodologie (§3), puis les ponts (§4).

## 1. NF, NFU, C₀ : le décor

La théorie **NF** (New Foundations, Quine, 1937) est une théorie des ensembles à compréhension **stratifiée** : l'ensemble {x | φ(x)} existe lorsque la formule φ respecte une stratification des variables par niveaux. Cette restriction est plus faible que la compréhension non restreinte de la théorie naïve, et plus étrange que celle de ZF : NF admet un ensemble universel et la négation de l'axiome du choix est un théorème. Sa cohérence est un problème **ouvert** depuis près de quatre-vingt-dix ans.

**NFU** est la variante avec ur-éléments (Jensen) ; l'adjonction d'axiomes d'infini et de choix y change la donne. **C₀** est une variante de la même famille, dont le dépôt source calibre des machines sur l'incohérence.

L'idée du **compte à rebours** est alors mécanique. Une machine de Turing à nombre d'états fini peut encoder la recherche de preuves d'une théorie T : elle énumère les dérivations et **s'arrête si et seulement si T dérive une contradiction**. Si l'équivalence « la machine s'arrête ⇔ T est incohérente » est elle-même un théorème — formalisé et audité — alors observer que la machine **ne s'arrête pas** établit Con(T). Autrement dit, le nombre d'états d'une machine devient une unité de mesure de la force d'une théorie : au-delà d'un seuil, la fonction du castor affairé dépasse toute fonction que T prouve totale.

C'est exactement le pont avec [07b](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb) : là-bas, le castor affairé mesurait la croissance ; ici, une **borne inférieure** sur sa valeur encode l'indépendance d'une théorie. Et c'est là qu'il faut tenir la distinction : **ce carnet ne parle pas de PA196**. L'indépendance visée ici est celle de **NF**, jamais une fusion avec Peano — le dépôt source écrit lui-même que le paquet à 184 états ne contient pas de nouveau théorème Busy Beaver interne à PA.

## 2. Le mécanisme sur un jouet déclaré

La vraie machine de 184 états encode la logique du premier ordre de NF sur une bande binaire : chaque instruction contribue à l'énumération des preuves. Reproduire cela ici serait une réimplémentation sans licence et sans preuve. À la place, nous construisons **notre propre** machine-jouet sur la logique propositionnelle : un contrôle fini qui énumère les résolutions d'une base de clauses et **s'arrête lorsque la clause vide est dérivée**. Le câblage est le même ; la théorie est minuscule, et c'est précisément ce qui rend le tout exécutable et vérifiable.

Le carnet n'implémente aucun artefact du dépôt source : il implémente l'**idée** du compte à rebours, avec un oracle indépendant (la fermeture par résolution) qui permet de contrôler la machine pas à pas.

In [2]:
use std::collections::HashSet;

fn est_tautologie(c: &[i64]) -> bool {
    c.iter().any(|l| c.contains(&-l))
}

fn resoudre(c1: &[i64], c2: &[i64], pivot: i64) -> Vec<i64> {
    let mut r: Vec<i64> = Vec::new();
    for l in c1 {
        if *l != pivot {
            r.push(*l);
        }
    }
    for l in c2 {
        if *l != -pivot {
            r.push(*l);
        }
    }
    r.sort_unstable();
    r.dedup();
    r
}

fn py_bool(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

fn fermeture(kb: &[Vec<i64>]) -> (Vec<Vec<i64>>, bool) {
    let mut tape: Vec<Vec<i64>> = Vec::new();
    let mut vus: HashSet<Vec<i64>> = HashSet::new();
    for c in kb {
        if !vus.contains(c) {
            vus.insert(c.clone());
            tape.push(c.clone());
        }
    }
    if tape.iter().any(|c| c.is_empty()) {
        return (tape, true);
    }
    let mut change = true;
    while change {
        change = false;
        let instantane = tape.clone();
        for i in 0..instantane.len() {
            for j in (i + 1)..instantane.len() {
                let c1 = instantane[i].clone();
                let c2 = instantane[j].clone();
                let pivots: Vec<i64> =
                    c1.iter().filter(|l| c2.contains(&-**l)).cloned().collect();
                for pivot in pivots {
                    let r = resoudre(&c1, &c2, pivot);
                    if r.is_empty() {
                        return (tape, true);
                    }
                    if !est_tautologie(&r) && !vus.contains(&r) {
                        vus.insert(r.clone());
                        tape.push(r);
                        change = true;
                    }
                }
            }
        }
    }
    (tape, false)
}

let k1: Vec<Vec<i64>> = vec![vec![1], vec![-1, 2], vec![-2]];
let k2: Vec<Vec<i64>> = vec![vec![1, 2], vec![-1, 2], vec![1, -2], vec![-1, -2]];
let k3: Vec<Vec<i64>> = vec![vec![1, 2], vec![-1, 2]];

for (nom, kb) in [("K1", &k1), ("K2", &k2), ("K3", &k3)] {
    let (clauses, refute) = fermeture(kb);
    println!(
        "{} : {} clause(s) dans la fermeture, réfutation = {}",
        nom,
        clauses.len(),
        py_bool(refute)
    );
}

// Note evcxr : la cellule se termine sur une liaison, sinon le kernel expose
// une valeur unitaire residuelle qui ne fait pas partie du resultat.
let _ = ();


K1 : 5 clause(s) dans la fermeture, réfutation = True


K2 : 8 clause(s) dans la fermeture, réfutation = True


K3 : 3 clause(s) dans la fermeture, réfutation = False


In [3]:
use std::collections::HashSet;

fn est_tautologie(c: &[i64]) -> bool {
    c.iter().any(|l| c.contains(&-l))
}

fn resoudre(c1: &[i64], c2: &[i64], pivot: i64) -> Vec<i64> {
    let mut r: Vec<i64> = Vec::new();
    for l in c1 {
        if *l != pivot {
            r.push(*l);
        }
    }
    for l in c2 {
        if *l != -pivot {
            r.push(*l);
        }
    }
    r.sort_unstable();
    r.dedup();
    r
}

fn py_bool(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

struct CompteARebours {
    tape: Vec<Vec<i64>>,
    vus: HashSet<Vec<i64>>,
    etat: String,
    i: usize,
    j: usize,
    pivots: Vec<i64>,
    kp: usize,
    ajout_passe: bool,
    pas: usize,
    halte: bool,
}

impl CompteARebours {
    fn new(base: &[Vec<i64>]) -> Self {
        let mut tape: Vec<Vec<i64>> = Vec::new();
        let mut vus: HashSet<Vec<i64>> = HashSet::new();
        for c in base {
            if !vus.contains(c) {
                vus.insert(c.clone());
                tape.push(c.clone());
            }
        }
        let mut m = CompteARebours {
            tape,
            vus,
            etat: "ENUM_I".to_string(),
            i: 0,
            j: 0,
            pivots: Vec::new(),
            kp: 0,
            ajout_passe: false,
            pas: 0,
            halte: false,
        };
        if m.tape.iter().any(|c| c.is_empty()) {
            m.halte = true;
            m.etat = "HALTE".to_string();
        }
        m
    }

    fn pas_once(&mut self) {
        if self.halte || self.etat == "BOULCLE" {
            return;
        }
        self.pas += 1;
        if self.etat == "ENUM_I" {
            if self.i >= self.tape.len() {
                self.etat = "FIN_PASSE".to_string();
            } else {
                self.j = self.i + 1;
                self.etat = "ENUM_J".to_string();
            }
        } else if self.etat == "ENUM_J" {
            if self.j >= self.tape.len() {
                self.i += 1;
                self.etat = "ENUM_I".to_string();
            } else {
                let c1 = self.tape[self.i].clone();
                let c2 = self.tape[self.j].clone();
                self.pivots = c1.iter().filter(|l| c2.contains(&-**l)).cloned().collect();
                self.kp = 0;
                self.etat = "RESOUT".to_string();
            }
        } else if self.etat == "RESOUT" {
            if self.kp >= self.pivots.len() {
                self.etat = "AVANCE_J".to_string();
            } else {
                let pivot = self.pivots[self.kp];
                self.kp += 1;
                let c1 = self.tape[self.i].clone();
                let c2 = self.tape[self.j].clone();
                let r = resoudre(&c1, &c2, pivot);
                if r.is_empty() {
                    self.halte = true;
                    self.etat = "HALTE".to_string();
                } else if !est_tautologie(&r) && !self.vus.contains(&r) {
                    self.vus.insert(r.clone());
                    self.tape.push(r);
                    self.ajout_passe = true;
                }
            }
        } else if self.etat == "AVANCE_J" {
            self.j += 1;
            self.etat = "ENUM_J".to_string();
        } else if self.etat == "FIN_PASSE" {
            if self.ajout_passe {
                self.ajout_passe = false;
                self.i = 0;
                self.etat = "ENUM_I".to_string();
            } else {
                self.etat = "BOULCLE".to_string();
            }
        }
    }
}

fn executer(mut m: CompteARebours, max_pas: usize) -> CompteARebours {
    while !m.halte && m.etat != "BOULCLE" && m.pas < max_pas {
        m.pas_once();
    }
    m
}

let k1: Vec<Vec<i64>> = vec![vec![1], vec![-1, 2], vec![-2]];
let k2: Vec<Vec<i64>> = vec![vec![1, 2], vec![-1, 2], vec![1, -2], vec![-1, -2]];
let k3: Vec<Vec<i64>> = vec![vec![1, 2], vec![-1, 2]];

for (nom, kb) in [("K1", &k1), ("K2", &k2), ("K3", &k3)] {
    let m = executer(CompteARebours::new(kb), 200_000);
    let verdict = if m.halte {
        "HALTE (réfutation dérivée)"
    } else if m.etat == "BOULCLE" {
        "tourne indéfiniment (fermeture épuisée sans clause vide)"
    } else {
        "budget épuisé sans verdict"
    };
    println!(
        "{} : {} après {} pas -- {} clause(s) sur la bande",
        nom,
        verdict,
        m.pas,
        m.tape.len()
    );
}

// Note evcxr : la cellule se termine sur une liaison, sinon le kernel expose
// une valeur unitaire residuelle qui ne fait pas partie du resultat.
let _ = ();


K1 : HALTE (réfutation dérivée) après 27 pas -- 5 clause(s) sur la bande


K2 : HALTE (réfutation dérivée) après 87 pas -- 8 clause(s) sur la bande


K3 : tourne indéfiniment (fermeture épuisée sans clause vide) après 36 pas -- 3 clause(s) sur la bande


In [4]:
use std::collections::HashSet;

fn est_tautologie(c: &[i64]) -> bool {
    c.iter().any(|l| c.contains(&-l))
}

fn resoudre(c1: &[i64], c2: &[i64], pivot: i64) -> Vec<i64> {
    let mut r: Vec<i64> = Vec::new();
    for l in c1 {
        if *l != pivot {
            r.push(*l);
        }
    }
    for l in c2 {
        if *l != -pivot {
            r.push(*l);
        }
    }
    r.sort_unstable();
    r.dedup();
    r
}

fn py_bool(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

struct CompteARebours {
    tape: Vec<Vec<i64>>,
    vus: HashSet<Vec<i64>>,
    etat: String,
    i: usize,
    j: usize,
    pivots: Vec<i64>,
    kp: usize,
    ajout_passe: bool,
    pas: usize,
    halte: bool,
}

impl CompteARebours {
    fn new(base: &[Vec<i64>]) -> Self {
        let mut tape: Vec<Vec<i64>> = Vec::new();
        let mut vus: HashSet<Vec<i64>> = HashSet::new();
        for c in base {
            if !vus.contains(c) {
                vus.insert(c.clone());
                tape.push(c.clone());
            }
        }
        let mut m = CompteARebours {
            tape,
            vus,
            etat: "ENUM_I".to_string(),
            i: 0,
            j: 0,
            pivots: Vec::new(),
            kp: 0,
            ajout_passe: false,
            pas: 0,
            halte: false,
        };
        if m.tape.iter().any(|c| c.is_empty()) {
            m.halte = true;
            m.etat = "HALTE".to_string();
        }
        m
    }

    fn pas_once(&mut self) {
        if self.halte || self.etat == "BOULCLE" {
            return;
        }
        self.pas += 1;
        if self.etat == "ENUM_I" {
            if self.i >= self.tape.len() {
                self.etat = "FIN_PASSE".to_string();
            } else {
                self.j = self.i + 1;
                self.etat = "ENUM_J".to_string();
            }
        } else if self.etat == "ENUM_J" {
            if self.j >= self.tape.len() {
                self.i += 1;
                self.etat = "ENUM_I".to_string();
            } else {
                let c1 = self.tape[self.i].clone();
                let c2 = self.tape[self.j].clone();
                self.pivots = c1.iter().filter(|l| c2.contains(&-**l)).cloned().collect();
                self.kp = 0;
                self.etat = "RESOUT".to_string();
            }
        } else if self.etat == "RESOUT" {
            if self.kp >= self.pivots.len() {
                self.etat = "AVANCE_J".to_string();
            } else {
                let pivot = self.pivots[self.kp];
                self.kp += 1;
                let c1 = self.tape[self.i].clone();
                let c2 = self.tape[self.j].clone();
                let r = resoudre(&c1, &c2, pivot);
                if r.is_empty() {
                    self.halte = true;
                    self.etat = "HALTE".to_string();
                } else if !est_tautologie(&r) && !self.vus.contains(&r) {
                    self.vus.insert(r.clone());
                    self.tape.push(r);
                    self.ajout_passe = true;
                }
            }
        } else if self.etat == "AVANCE_J" {
            self.j += 1;
            self.etat = "ENUM_J".to_string();
        } else if self.etat == "FIN_PASSE" {
            if self.ajout_passe {
                self.ajout_passe = false;
                self.i = 0;
                self.etat = "ENUM_I".to_string();
            } else {
                self.etat = "BOULCLE".to_string();
            }
        }
    }
}

fn executer(mut m: CompteARebours, max_pas: usize) -> CompteARebours {
    while !m.halte && m.etat != "BOULCLE" && m.pas < max_pas {
        m.pas_once();
    }
    m
}

struct Mt19937 {
    mt: [u32; 624],
    idx: usize,
}

impl Mt19937 {
    // initialisation identique a CPython : init_genrand(19650218) puis les deux
    // passes non lineaires de init_by_array (Modules/_randommodule.c).
    fn init_by_array(key: &[u32]) -> Self {
        const N: usize = 624;
        let mut mt = [0u32; N];
        mt[0] = 19650218;
        for i in 1..N {
            mt[i] = 1812433253u32
                .wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30))
                .wrapping_add(i as u32);
        }
        let mut i = 1usize;
        let mut j = 0usize;
        let mut k = if N > key.len() { N } else { key.len() };
        while k > 0 {
            k -= 1;
            mt[i] = (mt[i] ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525)))
                .wrapping_add(key[j])
                .wrapping_add(j as u32);
            i += 1;
            j += 1;
            if i >= N {
                mt[0] = mt[N - 1];
                i = 1;
            }
            if j >= key.len() {
                j = 0;
            }
        }
        let mut k2 = N - 1;
        while k2 > 0 {
            k2 -= 1;
            mt[i] = (mt[i] ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941)))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= N {
                mt[0] = mt[N - 1];
                i = 1;
            }
        }
        mt[0] = 0x80000000;
        Mt19937 { mt, idx: N }
    }

    fn genrand(&mut self) -> u32 {
        const N: usize = 624;
        const M: usize = 397;
        const MATRIX_A: u32 = 0x9908b0df;
        const UPPER: u32 = 0x80000000;
        const LOWER: u32 = 0x7fffffff;
        if self.idx >= N {
            for kk in 0..(N - M) {
                let y = (self.mt[kk] & UPPER) | (self.mt[kk + 1] & LOWER);
                self.mt[kk] = self.mt[kk + M] ^ (y >> 1) ^ if y & 1 != 0 { MATRIX_A } else { 0 };
            }
            for kk in (N - M)..(N - 1) {
                let y = (self.mt[kk] & UPPER) | (self.mt[kk + 1] & LOWER);
                self.mt[kk] = self.mt[kk - (N - M)] ^ (y >> 1) ^ if y & 1 != 0 { MATRIX_A } else { 0 };
            }
            let y = (self.mt[N - 1] & UPPER) | (self.mt[0] & LOWER);
            self.mt[N - 1] = self.mt[M - 1] ^ (y >> 1) ^ if y & 1 != 0 { MATRIX_A } else { 0 };
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c5680;
        y ^= (y << 15) & 0xefc60000;
        y ^= y >> 18;
        y
    }

    fn getrandbits(&mut self, k: u32) -> u32 {
        if k == 0 {
            return 0;
        }
        self.genrand() >> (32 - k)
    }

    // _randbelow de CPython : rejet tant que le tirage deborde.
    fn randbelow(&mut self, n: u32) -> u32 {
        if n == 0 {
            return 0;
        }
        let k = 32 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n {
            r = self.getrandbits(k);
        }
        r
    }
}

fn fermeture(kb: &[Vec<i64>]) -> (Vec<Vec<i64>>, bool) {
    let mut tape: Vec<Vec<i64>> = Vec::new();
    let mut vus: HashSet<Vec<i64>> = HashSet::new();
    for c in kb {
        if !vus.contains(c) {
            vus.insert(c.clone());
            tape.push(c.clone());
        }
    }
    if tape.iter().any(|c| c.is_empty()) {
        return (tape, true);
    }
    let mut change = true;
    while change {
        change = false;
        let instantane = tape.clone();
        for i in 0..instantane.len() {
            for j in (i + 1)..instantane.len() {
                let c1 = instantane[i].clone();
                let c2 = instantane[j].clone();
                let pivots: Vec<i64> =
                    c1.iter().filter(|l| c2.contains(&-**l)).cloned().collect();
                for pivot in pivots {
                    let r = resoudre(&c1, &c2, pivot);
                    if r.is_empty() {
                        return (tape, true);
                    }
                    if !est_tautologie(&r) && !vus.contains(&r) {
                        vus.insert(r.clone());
                        tape.push(r);
                        change = true;
                    }
                }
            }
        }
    }
    (tape, false)
}

fn kb_aleatoire(tirage: &mut Mt19937, n_clauses: usize) -> Vec<Vec<i64>> {
    let atomes: [i64; 3] = [1, 2, 3];
    let mut clauses: Vec<Vec<i64>> = Vec::new();
    for _ in 0..n_clauses {
        let taille = tirage.randbelow(3);
        let mut clause: Vec<i64> = Vec::new();
        for _ in 0..(taille as usize + 1) {
            let atome = atomes[tirage.randbelow(3) as usize];
            let signe = if tirage.randbelow(2) == 0 { 1 } else { -1 };
            clause.push(atome * signe);
        }
        clause.sort_unstable();
        clause.dedup();
        if !est_tautologie(&clause) && !clauses.contains(&clause) {
            clauses.push(clause);
        }
    }
    clauses
}

let mut rng = Mt19937::init_by_array(&[20261120]);

let mut accords = 0usize;
let mut desaccords: Vec<String> = Vec::new();
let mut indecis = 0usize;
let mut pas_max_refutation = 0usize;

for essai in 0..200 {
    let n_clauses = rng.randbelow(6) as usize + 1;
    let kb = kb_aleatoire(&mut rng, n_clauses);
    let (_, refute) = fermeture(&kb);
    let m = executer(CompteARebours::new(&kb), 50_000);
    if refute != m.halte {
        desaccords.push(format!(
            "essai {} : oracle={}, machine={}",
            essai,
            py_bool(refute),
            py_bool(m.halte)
        ));
    } else if refute && m.halte {
        accords += 1;
        if m.pas > pas_max_refutation {
            pas_max_refutation = m.pas;
        }
    } else if !refute && m.etat == "BOULCLE" {
        accords += 1;
    } else {
        indecis += 1;
    }
}

println!("accords oracle/machine : {}/200", accords);
println!("désaccords : {}", desaccords.len());
println!("indécis (budget épuisé avant point fixe) : {}", indecis);
println!("pas maximal observé pour une réfutation : {}", pas_max_refutation);
for ligne in desaccords.iter().take(5) {
    println!(" {}", ligne);
}

// Note evcxr : la cellule se termine sur une liaison, sinon le kernel expose
// une valeur unitaire residuelle qui ne fait pas partie du resultat.
let _ = ();


accords oracle/machine : 200/200


désaccords : 0


indécis (budget épuisé avant point fixe) : 0


pas maximal observé pour une réfutation : 140


### Lecture du résultat

La campagne contrôle la propriété du compte à rebours sur 200 bases tirées avec une graine fixée : quand l'oracle trouve une réfutation, la machine **halte** (et le nombre de pas est mesuré) ; quand l'oracle n'en trouve pas, la machine atteint le point fixe `BOULCLE` sans avoir halté. Aucun désaccord n'est attendu, et la catégorie « indécis » compte les tirages où le budget s'épuise avant le point fixe — une machine réelle, elle, n'a pas de budget : elle tourne.

Deux précautions rendent cette lecture honnête. D'abord, l'accord porte sur un **toy** : trois atomes, des clauses courtes, une fermeture finie — rien de tout cela ne dit quoi que ce soit de NF. Ensuite, la machine-jouet n'est **pas** la machine réelle : elle partage avec elle le câblage (énumération de dérivations, arrêt sur la clause vide, non-arrêt sur fermeture épuisée), pas l'encodage. C'est exactement le geste de la section 0 : distinguer ce qui est prouvé, calculé, et simplement illustré.

## 3. Les artefacts réels, épinglés (aucun vendoring)

Le dépôt `wlad-svennik/notes` (dernier push lu le 20/11/2026 à 11:19Z) publie, dossier par dossier :

| Dossier | Objet | Statut | Ce que le dépôt publie |
|---|---|---|---|
| `nf-bb210` | Machine à 210 états + développement Lean 4 externe sur NF | table + déclarations + audit enregistré | table, déclarations de théorèmes, audit, instantané 05/11 |
| `nf-bb192` | Machine-compte à rebours à 192 états : s'arrête ssi la NF du premier ordre prouve une contradiction | **théorème vérifié par Lean 4.35.0** sur la table littérale | `COUNTDOWN-PROOF.md`, `NF192Countdown.lean`, preuve informelle PDF, certificats hors ligne |
| `nf-bb185` | Machine à 185 états, même équivalence | **théorème Lean non borné** `NFMachine185.halts_iff_nf_refutation` | audit d'axiomes (`theorem-axioms.txt`) : 3 axiomes standard + 11 vérifications natives ; traces de simulation, volumes dans le dépôt |
| `nf-bb184` | Machine à 184 états, NF pure de Quine | **calcul reproductible** | table `NF184.tm` (SHA-256 `7764cc84…8798`), 7 794 théorèmes signés, 142 608 essais / 88 555 587 pas binaires, 2 809 instructions émettrices, point de contrôle 1,84 Mo, `SHA256SUMS`, scripts de rejeu |
| `c0-198` | Deux machines à 198 états calibrées sur l'incohérence de C₀ | **borne conditionnelle** | rapports de preuve, points de contrôle reproductibles, rapport de réverse-interprétation (Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity)) |

**Ce que « épinglé » veut dire ici** : chaque affirmation renvoie à une révision du dépôt (par exemple `1297cce204` pour le paquet 184, `e0a8130149` et `bfc960b072` pour le 185, `299f86fd7c` et `a006d3abba` pour le 192, `33772f530c` et `9823a07d6b` pour les interprétations, `0a0c8fa599` pour C₀). Le carnet ne recopie ni table ni fichier ; il cite la révision et laisse la vérification au dépôt. La question de licence est ouverte (`wlad-svennik/notes#1`) : tant qu'elle ne reçoit pas de réponse, **zéro vendoring** reste la règle, et chaque greffe se limite à citation + consommation épinglée ou exécution de l'outil cité.

**La chaîne de compression documentée** est `210 → 201 → 192 → 185 → 184`. Le dépôt en expose la méthodologie : table à marqueur variable, testeur `D(x,y)=(J(Kx))y` et formule `((D(x,y)=x)→(D(x,x)=x))→y=x`, réordonnancement des huit rangées, états décodeurs partagés — et **les échecs conservés** (`WORK_NOTES.md` garde un contre-modèle à trois éléments pour `collapsed183`). Le prochain calcul ne fait pas de la physique : il mesure la compression elle-même.

In [5]:

let chaine: [(&str, u32); 5] = [
    ("nf-bb210", 210),
    ("étape intermédiaire", 201),
    ("nf-bb192", 192),
    ("nf-bb185", 185),
    ("nf-bb184", 184),
];

println!("chaîne de compression documentée par le dépôt (états) :");
let mut precedent_nom = chaine[0].0;
let mut precedent = chaine[0].1;
for k in 1..chaine.len() {
    let (nom, n) = chaine[k];
    let delta = precedent - n;
    let part = (delta as f64) / (precedent as f64) * 100.0;
    println!(
        "  {:20} {:3} -> {:20} {:3} : -{} état(s), {:.2}% de la taille précédente",
        precedent_nom,
        precedent,
        nom,
        n,
        delta,
        part
    );
    precedent_nom = nom;
    precedent = n;
}

let total = chaine[0].1 - chaine[4].1;
let part_totale = (total as f64) / (chaine[0].1 as f64) * 100.0;
println!("total : -{} états ({:.2}%) de 210 à 184", total, part_totale);


chaîne de compression documentée par le dépôt (états) :


  nf-bb210             210 -> étape intermédiaire  201 : -9 état(s), 4.29% de la taille précédente


  étape intermédiaire  201 -> nf-bb192             192 : -9 état(s), 4.48% de la taille précédente


  nf-bb192             192 -> nf-bb185             185 : -7 état(s), 3.65% de la taille précédente


  nf-bb185             185 -> nf-bb184             184 : -1 état(s), 0.54% de la taille précédente


total : -26 états (12.38%) de 210 à 184


### Lecture de la chaîne

Le calcul ci-dessus montre que la compression est **modeste en pourcentage** : vingt-six états retirés sur deux cent dix, soit un peu plus de douze pour cent. Ce qui compte n'est pas l'ampleur, c'est ce que chaque état retiré **achète** : à chaque étape, la même équivalence d'arrêt doit rester un théorème, sur une table plus petite. Le dépôt documente les échecs intermédiaires, et c'est le point méthodologique du carnet : un contre-modèle à trois éléments pour `collapsed183` est une **information**, pas un déchet — il dit exactement ce qui casse quand on retire un état de trop.

La leçon est transposable : dans toute compression de machine, l'invariant qui doit survivre n'est pas la forme de la table, c'est le **théorème d'équivalence** qui la relie à la théorie. Les états décodeurs partagés sont un levier ; la vérification Lean est le juge.

In [6]:

fn sha256_hex(data: &[u8]) -> String {
    const K: [u32; 64] = [0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5, 0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174, 0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da, 0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967, 0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85, 0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070, 0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3, 0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2];
    let mut h: [u32; 8] = [0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a, 0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19];
    let mut msg = data.to_vec();
    let bitlen = (data.len() as u64) * 8;
    msg.push(0x80);
    while msg.len() % 64 != 56 {
        msg.push(0);
    }
    msg.extend_from_slice(&bitlen.to_be_bytes());
    let mut w = [0u32; 64];
    for chunk in msg.chunks(64) {
        for i in 0..16 {
            w[i] = u32::from_be_bytes([chunk[4 * i], chunk[4 * i + 1], chunk[4 * i + 2], chunk[4 * i + 3]]);
        }
        for i in 16..64 {
            let s0 = w[i - 15].rotate_right(7) ^ w[i - 15].rotate_right(18) ^ (w[i - 15] >> 3);
            let s1 = w[i - 2].rotate_right(17) ^ w[i - 2].rotate_right(19) ^ (w[i - 2] >> 10);
            w[i] = w[i - 16]
                .wrapping_add(s0)
                .wrapping_add(w[i - 7])
                .wrapping_add(s1);
        }
        let (mut a, mut b, mut c, mut d) = (h[0], h[1], h[2], h[3]);
        let (mut e, mut f, mut g, mut hh) = (h[4], h[5], h[6], h[7]);
        for i in 0..64 {
            let s1 = e.rotate_right(6) ^ e.rotate_right(11) ^ e.rotate_right(25);
            let ch = (e & f) ^ ((!e) & g);
            let t1 = hh
                .wrapping_add(s1)
                .wrapping_add(ch)
                .wrapping_add(K[i])
                .wrapping_add(w[i]);
            let s0 = a.rotate_right(2) ^ a.rotate_right(13) ^ a.rotate_right(22);
            let maj = (a & b) ^ (a & c) ^ (b & c);
            let t2 = s0.wrapping_add(maj);
            hh = g;
            g = f;
            f = e;
            e = d.wrapping_add(t1);
            d = c;
            c = b;
            b = a;
            a = t1.wrapping_add(t2);
        }
        h[0] = h[0].wrapping_add(a);
        h[1] = h[1].wrapping_add(b);
        h[2] = h[2].wrapping_add(c);
        h[3] = h[3].wrapping_add(d);
        h[4] = h[4].wrapping_add(e);
        h[5] = h[5].wrapping_add(f);
        h[6] = h[6].wrapping_add(g);
        h[7] = h[7].wrapping_add(hh);
    }
    h.iter().map(|x| format!("{:08x}", x)).collect()
}

let formule_jouet = "D(x,y)=(J(Kx))y; ((D(x,y)=x)->(D(x,x)=x))->y=x";
let empreinte = sha256_hex(formule_jouet.as_bytes());

println!("formule-jouet : {}", formule_jouet);
println!("empreinte SHA-256 : {}", empreinte);
println!("longueur : {} caractères hexadécimaux", empreinte.len());
println!();
println!("Empreinte publiée du paquet 184 : 7764cc84...8798 (citée du dépôt, révision 1297cce204).");
println!("Elle n'est PAS re-vérifiable ici : la table NF184.tm n'est pas vendorée (aucune licence).");
println!("Citée ne veut pas dire vérifiée -- c'est la règle de la section 0.");


formule-jouet : D(x,y)=(J(Kx))y; ((D(x,y)=x)->(D(x,x)=x))->y=x


empreinte SHA-256 : 8c83e4ed6e2eb9e05f81d9f29090af521bfc3af2da20924f9b2a10b409896719


longueur : 64 caractères hexadécimaux


Empreinte publiée du paquet 184 : 7764cc84...8798 (citée du dépôt, révision 1297cce204).


Elle n'est PAS re-vérifiable ici : la table NF184.tm n'est pas vendorée (aucune licence).


Citée ne veut pas dire vérifiée -- c'est la règle de la section 0.


### Ce que ce carnet ne peut pas faire

- **Il ne vérifie pas la table de 184 états.** Sans vendoring, l'empreinte publiée reste une citation ; le `SHA256SUMS` du dépôt source est l'instrument qui la contrôle, et il s'exécute là-bas.
- **Il ne prouve rien sur NF.** La cohérence de NF reste ouverte ; ce que les machines déplacent, c'est la **forme** de la question : « NF est-elle cohérente ? » devient « cette machine s'arrête-t-elle ? ». Le déplacement est utile parce qu'il rend l'énoncé fini et vérifiable, pas parce qu'il le résout.
- **Il ne fusionne pas les familles.** PA196 (carnet 08d, Greffe A) est une autre théorie, un autre dépôt, un autre statut. Toute prose qui mélange PA et NF rend les deux résultats inutilisables.

## 4. Les sept théories interprétables (pont vers formal_logic)

Le dossier `nf-bb192/interpretations` publie un index théorème par théorème (`SET-THEORY-PROOFS.md`) et une archive `nf-interp.zip` couvrant **sept** théories : NF, FunctionNF, CombNF, M, MAC, M+H et NFU + Infinity. C'est un objet d'une autre nature que les machines : une carte d'**interprétabilité** et d'**équiconsistance**, avec ses fichiers Lean originaux et leurs expositions (inventaire : SET-THEORY-PROOFS.md).

Le pont est direct vers la série `SymbolicAI/Lean` (formal_logic) : l'index cite la formulation exacte de H utilisée, et la question « quelles théories sont équiconsistantes ? » est le pendant sémantique de la question « quelles machines s'arrêtent ? ». Le carnet s'arrête ici volontairement : la consommation de cet index est un jalon distinct (Jalon 3 de [#27537](https://github.com/jsboige/CoursIA/issues/27537)), qui exigera de lire la copie archivée au gisement avant d'écrire — la règle bibliographique du dépôt.

## 5. Exercices

Trois exercices, stubs exécutables sans erreur volontaire (C.1) : le carnet s'exécute de bout en bout même non complétés.

### Exercice 1 — La cible du compte à rebours

Notre machine s'arrête sur la **clause vide**, c'est-à-dire sur une contradiction. Généralisez-la : une machine qui s'arrête sur une clause `cible` donnée, et seulement sur elle. La propriété à vérifier ensuite est la même, décalée d'un cran : `halte <=> cible appartient à la fermeture(kb)`.

*Indice* : la clause vide est le cas particulier `cible = []` (le vecteur vide) ; en Rust, l'absence de réponse se rend par `None`. Le changement porte sur le test d'arrêt dans l'état `RESOUT`, pas sur l'énumération.

In [7]:
use std::collections::HashSet;

fn est_tautologie(c: &[i64]) -> bool {
    c.iter().any(|l| c.contains(&-l))
}

fn resoudre(c1: &[i64], c2: &[i64], pivot: i64) -> Vec<i64> {
    let mut r: Vec<i64> = Vec::new();
    for l in c1 {
        if *l != pivot {
            r.push(*l);
        }
    }
    for l in c2 {
        if *l != -pivot {
            r.push(*l);
        }
    }
    r.sort_unstable();
    r.dedup();
    r
}

fn py_bool(b: bool) -> &'static str {
    if b {
        "True"
    } else {
        "False"
    }
}

let k3: Vec<Vec<i64>> = vec![vec![1, 2], vec![-1, 2]];

// TODO etudiant : renvoyer (halte, pas) pour une machine qui s'arrete sur `cible`.
// Indice : dupliquer CompteARebours en parametrant le test d'arret.
// Verification attendue : comparer halte_si_derive(K3, vec![2]) a fermeture(K3).
fn halte_si_derive(_kb: &[Vec<i64>], _cible: &[i64], _max_pas: usize) -> (Option<bool>, usize) {
    // TODO etudiant
    (None, 0)
}

let (halte, nb_pas) = halte_si_derive(&k3, &[2], 200_000);
let _sortie = match halte {
    None => println!("Exercice à compléter"),
    Some(h) => println!("K3 dérive {{2}} : halte={} en {} pas", py_bool(h), nb_pas),
};


Exercice à compléter


### Exercice 2 — Rétrécir sans perdre l'équivalence

La chaîne réelle descend de 210 à 184 états. Écrivez le profil de réduction : pour une suite d'entiers, le pourcentage retiré à chaque étape et le total. Puis répondez en une phrase, dans une cellule markdown : quel invariant doit survivre à **chaque** compression, et pourquoi un contre-modèle conservé (le `collapsed183` à trois éléments) vaut-il un résultat ?

*Indice* : `profil_reduction(&[210, 201, 192, 185, 184])` doit renvoyer quatre valeurs, et la somme des retraits doit redonner vingt-six états.

In [8]:

// TODO etudiant : renvoyer les reductions successives, en pourcentage du precedent.
// Indice : quatre valeurs pour cinq etats ; le total doit etre coherent.
fn profil_reduction(_etats: &[u32]) -> Option<Vec<f64>> {
    // TODO etudiant
    None
}

let profil = profil_reduction(&[210, 201, 192, 185, 184]);
let _sortie = match profil {
    None => println!("Exercice à compléter"),
    Some(p) => println!("profil : {:?}", p),
};


Exercice à compléter


### Exercice 3 — Classer un résultat avant de le citer

On vous donne trois affirmations. Classez chacune dans l'un des trois statuts de la section 0 : `formalise-audite`, `calcul-reproductible` ou `borne-conditionnelle`.

a) « La machine à 185 états s'arrête si et seulement si NF est réfutable, et l'audit ne liste que les axiomes standard. »
b) « Un simulateur indépendant a exécuté 142 608 essais sur la table à 184 états, sans divergence. »
c) « Si NF est cohérente, alors NFU + Infinity est cohérente. »

*Indice* : demandez-vous, pour chaque énoncé, **qui** peut le vérifier et **avec quel artefact** — c'est la question qui distingue un statut d'un autre.

In [9]:

// TODO etudiant : renvoyer "formalise-audite", "calcul-reproductible" ou "borne-conditionnelle".
fn classer(_enonce: &str) -> Option<String> {
    // TODO etudiant
    None
}

let enonces = [
    "La machine à 185 états s'arrête si et seulement si NF est réfutable (audit d'axiomes publié).",
    "Un simulateur indépendant a exécuté 142 608 essais sur la table à 184 états.",
    "Si NF est cohérente, alors NFU + Infinity est cohérente.",
];
for enonce in enonces.iter() {
    let _sortie = match classer(enonce) {
        None => println!("Exercice à compléter"),
        Some(v) => {
            let extrait: String = enonce.chars().take(60).collect();
            println!("{} : {}...", v, extrait);
        }
    };
}

// Note evcxr : la cellule se termine sur une liaison, sinon le kernel expose
// une valeur unitaire residuelle qui ne fait pas partie du resultat.
let _ = ();


Exercice à compléter


Exercice à compléter


Exercice à compléter


## 6. Conclusion et ponts

Le compte à rebours est une **traduction** : il transforme une question sur une théorie (« NF est-elle cohérente ? ») en une question sur une machine finie (« s'arrête-t-elle ? »). Trois exigences en découlent, et le carnet les a tenues séparément :

1. **L'équivalence doit être un théorème.** C'est le statut des machines 185 et 192, auditées. Le jouet de la section 2 n'a pas ce statut : il illustre le câblage, il ne prouve rien.
2. **La table doit être reproductible.** C'est le statut du paquet 184 : empreinte, simulateur indépendant, points de contrôle. Citer une empreinte n'est pas la vérifier.
3. **Une borne se cite avec sa condition.** C'est le statut de C₀-198 : Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity), condition comprise.

Les ponts naturels sont la série `SymbolicAI/Lean` (l'index d'interprétabilité des sept théories, Jalon 3) et le carnet 07b (le castor affairé comme fonction de croissance, dont ces machines calculent des valeurs inférieures). Le Jalon 2 de [#27537](https://github.com/jsboige/CoursIA/issues/27537) — un lac consommateur `nf_bb_lean` épinglé sur NF185 — viendra quand la fenêtre de build bornée sera libre ; ce carnet prépare sa prose sans emprunter sa dette.

## 7. Références

**Source primaire — dépôt `wlad-svennik/notes`** (lu firsthand le 20/11/2026, dernier push 11:19Z ; révisions épinglées citées en section 3). Aucune licence détectée dans le dépôt ; demande de clarification ouverte : `wlad-svennik/notes#1`. Aucun contenu recopié.

**Copies archivées au gisement** (bibliography-hygiene, identités vérifiées page 1), rayon `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\` :

- `2026 - Svennik (wlad) - 192-state PA bound for non-regularity (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - 192-state countdown machine and NF (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - tangled type theory and Con NF (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - 201-state Zermelo (github wlad-svennik-notes, sans licence).pdf`

**Références de fond, non consommées dans ce carnet** : Quine, *New Foundations for Mathematical Logic* (1937) — compréhension stratifiée ; Jensen, résultats de cohérence de NFU ; le carnet [Complexity-07b](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb) pour le castor affairé et Tibor Radó (1962) ; [Complexity-08d](https://github.com/jsboige/CoursIA/pull/27483) (Greffe A) pour PA196, qui n'est **pas** fusionné avec ce carnet.

*Greffe D de [#27537](https://github.com/jsboige/CoursIA/issues/27537), EPIC [#20607](https://github.com/jsboige/CoursIA/issues/20607).*